# Tipos de Aprendizaje Automático

En este notebook exploraremos los tres paradigmas fundamentales del Machine Learning con ejemplos aplicados a la industria AEC.

## 1. Aprendizaje Supervisado

El modelo aprende de **ejemplos etiquetados**: pares (entrada, salida esperada).

### Analogía AEC
Imagina que tienes un historial de 500 proyectos con sus costos reales. Le das al modelo las características (área, número de pisos, tipo de estructura, ubicación) y el costo final. El modelo aprende la relación y puede **predecir el costo** de un proyecto nuevo.

**Tipos:**
- **Regresión**: predice un valor continuo (costo, plazo, resistencia).
- **Clasificación**: predice una categoría (tipo de falla, categoría de riesgo).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# --- Ejemplo: Predecir costo de obra según área construida ---
# Datos ficticios de proyectos (área en m², costo en millones COP)
np.random.seed(42)
areas = np.array([120, 250, 480, 600, 800, 1100, 1500, 2000, 2500, 3200])
costos = areas * 2.8 + np.random.normal(0, 80, size=len(areas))  # ~2.8M COP/m²

# Entrenar modelo de regresión lineal
modelo = LinearRegression()
modelo.fit(areas.reshape(-1, 1), costos)

# Predicción
area_nueva = np.array([[1800]])
costo_predicho = modelo.predict(area_nueva)

# Visualización
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(areas, costos, color='steelblue', s=80, label='Proyectos históricos')
ax.plot(areas, modelo.predict(areas.reshape(-1, 1)), color='tomato', 
        linewidth=2, label=f'Modelo: costo ≈ {modelo.coef_[0]:.1f} × área + {modelo.intercept_:.0f}')
ax.scatter(*area_nueva.flatten(), costo_predicho, color='green', s=150, 
           marker='*', zorder=5, label=f'Predicción: {area_nueva[0,0]}m² → {costo_predicho[0]:.0f}M COP')
ax.set_xlabel('Área construida (m²)')
ax.set_ylabel('Costo total (millones COP)')
ax.set_title('Aprendizaje Supervisado — Regresión Lineal')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### ¿Qué acaba de pasar?

1. Le dimos al modelo **datos históricos** (área → costo).
2. El modelo encontró la **relación lineal** entre ambas variables.
3. Ahora puede **predecir** el costo para un área que nunca vio.

Esto es la esencia del aprendizaje supervisado: aprender de ejemplos pasados para predecir el futuro.

## 2. Aprendizaje No Supervisado

El modelo encuentra **patrones ocultos** en datos **sin etiquetas**.

### Analogía AEC
Tienes datos de 200 edificaciones (área, pisos, tipo de cimentación, zona sísmica) pero no sabes cómo agruparlas. El modelo descubre que naturalmente se forman 3 grupos: edificaciones residenciales pequeñas, edificios de oficinas medianos y torres altas — sin que tú le hayas dicho esas categorías.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# --- Ejemplo: Agrupar edificaciones por características ---
np.random.seed(42)

# Grupo 1: Casas pequeñas (área baja, pocos pisos)
g1 = np.random.normal(loc=[150, 2], scale=[40, 0.5], size=(30, 2))
# Grupo 2: Edificios medianos
g2 = np.random.normal(loc=[800, 8], scale=[150, 2], size=(30, 2))
# Grupo 3: Torres altas
g3 = np.random.normal(loc=[2500, 25], scale=[400, 5], size=(30, 2))

datos = np.vstack([g1, g2, g3])
datos[:, 1] = np.clip(datos[:, 1], 1, None)  # mínimo 1 piso

# Escalar datos y aplicar K-Means
scaler = StandardScaler()
datos_scaled = scaler.fit_transform(datos)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(datos_scaled)

# Visualización
colores = ['#2ecc71', '#3498db', '#e74c3c']
nombres = ['Residencial bajo', 'Edificio mediano', 'Torre alta']

fig, ax = plt.subplots(figsize=(8, 5))
for i in range(3):
    mask = clusters == i
    ax.scatter(datos[mask, 0], datos[mask, 1], c=colores[i], 
               s=60, alpha=0.7, label=nombres[i])

ax.set_xlabel('Área construida (m²)')
ax.set_ylabel('Número de pisos')
ax.set_title('Aprendizaje No Supervisado — Clustering K-Means')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### ¿Qué acaba de pasar?

1. Le dimos datos **sin etiquetas** — solo área y número de pisos.
2. K-Means encontró **3 agrupaciones naturales** en los datos.
3. Esto es útil para segmentar portafolios de proyectos, identificar tipologías constructivas, o detectar anomalías.

## 3. Aprendizaje por Refuerzo

El modelo aprende por **prueba y error**, recibiendo recompensas o penalizaciones.

### Analogía AEC
Imagina un robot que programa actividades de obra. Cada vez que asigna una cuadrilla eficientemente (sin tiempos muertos) recibe una **recompensa**. Cada vez que genera un conflicto de recursos, recibe una **penalización**. Con el tiempo, aprende a programar mejor.

```
┌─────────┐     acción      ┌─────────────┐
│  Agente │ ───────────────→│  Ambiente    │
│ (IA)    │ ←───────────────│  (Obra)      │
└─────────┘  estado +       └─────────────┘
             recompensa
```

**Aplicaciones en AEC:**
- Optimización de rutas de grúa torre
- Programación automática de obra
- Control de sistemas HVAC en edificios inteligentes
- Robots autónomos en obra (drones de inspección)

> El aprendizaje por refuerzo es más complejo de implementar y requiere un *entorno de simulación*. Por ahora es suficiente entender el concepto — en capítulos posteriores veremos ejemplos prácticos.

## Resumen comparativo

| Tipo | Datos | Objetivo | Ejemplo AEC |
|------|-------|----------|-------------|
| **Supervisado** | Etiquetados (X → y) | Predecir | Estimar costo según área |
| **No supervisado** | Sin etiquetas (solo X) | Descubrir patrones | Agrupar tipologías de edificaciones |
| **Refuerzo** | Interacción con entorno | Optimizar decisiones | Programar secuencia de obra |

## Ejercicio propuesto

Modifica el ejemplo de regresión lineal:
1. Agrega una segunda variable: **número de pisos**.
2. Genera datos donde el costo dependa del área Y los pisos.
3. Entrena el modelo y compara el error con el modelo de una sola variable.

*Pista: usa `areas` y `pisos` como columnas de una matriz X de dos columnas.*